# Sta-RU Video Dubbing — Edge-TTS (Multi-language, YouTube-resilient)

Variant of the Edge notebook that **squeezes a Colab session to the max**:

- The expensive YouTube work (download + Demucs ambient) happens **once per video** and is cached, so producing the same video in another language reuses the cache — no YouTube, no Demucs, just TTS + mux.
- **Breadth-first:** all of the *primary* language first (your priority). If YouTube's bot wall kills downloads mid-batch, it stops downloading and dubs the **already-downloaded** videos into the *secondary* languages (default voice each).
- A genuine TTS failure now **aborts that video** instead of shipping it with a silent gap.

Sandbox copy — the production `Sta_RU_Dubbing_Edge.ipynb` is untouched.

## 1. Setup — install deps, clone repo & YouTube preflight

One cell does it all: installs dependencies, clones (or updates) the Sta-RU repo, and runs the YouTube **preflight** (the public canary *"Me at the zoo"*) so you find out up front whether this VM is bot-walled.

- If it prints **OK**, continue to step 2.
- If it prints **BLOCKED**, re-run it (sometimes passes on retry); if it persists, run the **Cookies** cell below, then re-run this one.

In [ ]:
import os, sys

# 1) System + Python dependencies
!apt-get -qq install -y ffmpeg rubberband-cli
!pip install -q edge-tts nest-asyncio yt-dlp srt soundfile numpy scipy demucs deep-translator pyrubberband ipywidgets pandas

# 2) Clone (or update) the repo and put colab/ on the import path
REPO_DIR = '/content/Sta-RU'
BRANCH = 'claude/compassionate-knuth-M2llL'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b $BRANCH https://github.com/lazy-money/sta-ru.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull --quiet
_colab = os.path.join(REPO_DIR, 'colab')
if _colab not in sys.path:
    sys.path.insert(0, _colab)
print('Repo ready at', REPO_DIR, '(branch:', BRANCH, ')')

# 3) Preflight: can THIS VM reach YouTube? Same player-client logic as the
#    real pipeline, so if this passes the batch will too. (Import happens
#    here, after the clone + sys.path, on purpose.)
from batch_dub import preflight_youtube
CANARY_URL = 'https://www.youtube.com/watch?v=jNQXAC9IVRw'
print('\nProbing YouTube access with the canary:', CANARY_URL)
print('(tries several player clients; may take a few seconds)')
_res = preflight_youtube(CANARY_URL)
if _res['ok']:
    print("\nOK: YouTube reachable -- passed with client '%s'." % _res['client'])
    print('The VM is not blocked. Continue from step 2 (Mount Drive).')
elif _res['client'] is None:
    print('\nBLOCKED: YouTube bot check (cloud IP). Last error:')
    print('  ', _res['error'])
    print('   -> Re-run this cell (it sometimes passes on a retry).')
    print('   -> Change IP: Runtime > Disconnect and delete runtime, reconnect and retry.')
    print('   -> If it persists, run the COOKIES cell below (last resort), then re-run this.')
else:
    print('\nODD: the canary failed with an error that is NOT the bot check:')
    print('  ', _res['error'])


### Cookies — last resort (optional)

Run the next cell **only** if the Setup preflight printed `BLOCKED`. Export `cookies.txt` from a logged-in (secondary) YouTube account with the *"Get cookies.txt LOCALLY"* extension, upload it when prompted, then re-run the Setup cell above to confirm it passes.

In [ ]:
# COOKIES (last resort) -- run this cell ONLY if the preflight above printed BLOCKED.
# 1) Install the "Get cookies.txt LOCALLY" extension (Chrome/Firefox), open
#    youtube.com logged in with a SECONDARY account, and export cookies.txt.
# 2) Run this cell and upload the file when prompted.
import batch_dub
from google.colab import files as _gfiles

_up = _gfiles.upload()
if _up:
    _name = next(iter(_up))
    _path = '/content/cookies.txt'
    with open(_path, 'wb') as _f:
        _f.write(_up[_name])
    batch_dub.YTDLP_COOKIES = _path
    print('Cookies loaded from %s -> %s' % (_name, _path))
    print('Now re-run the preflight above to confirm it passes.')
else:
    print('No file uploaded. (Run this cell only if the preflight failed.)')

## 2. Mount Google Drive (optional)

Run this **after** the preflight passes — mounting is slow, so there's no point doing it until you know the VM can reach YouTube.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Provide YouTube URLs

One per line. Use `2: https://...` to force a specific N#.

In [ ]:
import ipywidgets as W
from IPython.display import display

url_box = W.Textarea(
    value='',
    placeholder=('Paste one YouTube URL per line.\n'
                 'https://www.youtube.com/watch?v=...\n'
                 '\nOr force N#:\n'
                 '2: https://www.youtube.com/watch?v=...'),
    description='URLs:',
    layout=W.Layout(width='100%', height='180px'),
)
display(url_box)

## 4. Upload subtitle files

Format: `{N#}-{LANG}.srt` (e.g. `1-EN.srt`, `47-ES.srt`, `47-DE.srt`). Multi-select supported. **Upload the SRTs for every language you select in step 5** — the primary and each secondary.

In [ ]:
from google.colab import files
import os, re

SRT_DIR = '/content/srts'
os.makedirs(SRT_DIR, exist_ok=True)
_srts = files.upload()
_pattern = re.compile(r'^(\d+)-([A-Z]{2})\.srt$')
loaded = []
for name, content in _srts.items():
    with open(os.path.join(SRT_DIR, name), 'wb') as f:
        f.write(content)
    m = _pattern.match(name)
    if m:
        loaded.append((int(m.group(1)), m.group(2)))
print(f'\nLoaded {len(loaded)} SRTs')
by_lang = {}
for n, lg in loaded:
    by_lang.setdefault(lg, []).append(n)
for lg, ns in by_lang.items():
    print(f'  {lg}: N# {sorted(ns)}')

## 5. Options

In [ ]:
import ipywidgets as W
import os
from IPython.display import display

_drive_mounted = os.path.ismount('/content/drive')
_default_out_mode = 'drive' if _drive_mounted else 'local'
_default_out_lang = 'EN'
_default_out_path = (f'/content/drive/MyDrive/Dubbing/{_default_out_lang}' if _drive_mounted
                     else f'/content/output/{_default_out_lang}')
from batch_dub_edge import DEFAULT_VOICES, VOICE_CHOICES

LANG_OPTIONS = list(DEFAULT_VOICES.keys())

lang_w            = W.Dropdown(options=LANG_OPTIONS, value='EN', description='Target language:')
gender_w          = W.RadioButtons(options=[('Male', 'M'), ('Female', 'F')], value='M', description='Voice gender:')
def _voice_options(lang, gender):
    vs = VOICE_CHOICES.get(lang, {}).get(gender) or [DEFAULT_VOICES[lang][gender]]
    return [(v + ' (default)' if i == 0 else v, v) for i, v in enumerate(vs)]

voice_custom_w    = W.Dropdown(options=_voice_options('EN', 'M'),
                               description='Voice:', layout=W.Layout(width='600px'))
pitch_w           = W.IntSlider(value=-5, min=-20, max=20, step=1, description='Pitch (Hz):')
range_w           = W.Text(value='all', description='Range (N#):', placeholder="'all', '1-10', '47'")
remove_voice_w    = W.Checkbox(value=True,  description='Remove original voice (Demucs)')
dynamic_dur_w     = W.Checkbox(value=True, description='Dynamic duration (stretch video to fit dubbing)')
skip_silent_w     = W.Checkbox(value=True,  description='Skip TTS where the original speaker is silent')
burn_subs_w       = W.Checkbox(value=False, description='Burn subtitles into the video')
translate_title_w = W.Checkbox(value=True, description='Translate title to target language')
hq_demucs_w       = W.Checkbox(value=False, description='Higher-quality vocal removal (mdx_extra) — slower')
allow_no_amb_w    = W.Checkbox(value=False, description='Continue without ambient if Demucs fails')
ambient_gain_w    = W.FloatSlider(value=3.5, min=1.0, max=8.0, step=0.25,
                                  description='Ambient gain:', readout_format='.2f')

out_mode_w = W.RadioButtons(
    options=[('Save to Google Drive', 'drive'), ('Save locally', 'local')],
    value=_default_out_mode, description='Output:',
)
out_path_w = W.Text(value=_default_out_path, description='Output dir:',
                    layout=W.Layout(width='600px'))
cache_path_w = W.Text(value='/tmp/sta-ru-cache', description='Cache dir:',
                      layout=W.Layout(width='600px'))

def _sync(_=None):
    lg = lang_w.value
    out_path_w.value = (f'/content/drive/MyDrive/Dubbing/{lg}' if out_mode_w.value == 'drive'
                        else f'/content/output/{lg}')
lang_w.observe(_sync, names='value')
out_mode_w.observe(_sync, names='value')

def _refresh_voices(_=None):
    voice_custom_w.options = _voice_options(lang_w.value, gender_w.value)
    voice_custom_w.index = 0   # select the language/gender default
lang_w.observe(_refresh_voices, names='value')
gender_w.observe(_refresh_voices, names='value')

# --- Secondary 'Other lang dub' cascade (opportunistic; cached videos only) ---
# Pick the primary above. Choosing a language here reveals the next 'Other dub'
# selector below it; each one hides languages already picked (and the primary),
# so you can't repeat one. Selection order = priority order of the extra dubs.
_sec_label = W.HTML('<b>Other lang dub</b> (optional &mdash; default voice each; '
                    'run on already-downloaded videos after the primary)')
sec_box = W.VBox([])
_ALL_LANGS = list(DEFAULT_VOICES.keys())
_rebuilding = {'on': False}
def _sec_opts(exclude):
    avail = [l for l in _ALL_LANGS if l not in exclude]
    return [('\u2014 none \u2014', None)] + [(l, l) for l in avail]
def chosen_secondaries():
    out = []
    for dd in sec_box.children:
        v = dd.value
        if v and v != lang_w.value and v not in out:
            out.append(v)
    return out
def _rebuild_cascade(_=None):
    if _rebuilding['on']:
        return
    _rebuilding['on'] = True
    try:
        picks = chosen_secondaries()
        used = [lang_w.value] + picks
        rows = []
        for i, p in enumerate(picks):
            dd = W.Dropdown(options=_sec_opts([u for u in used if u != p]), value=p,
                            description=f'Other dub {i+1}:', layout=W.Layout(width='320px'))
            dd.observe(_rebuild_cascade, names='value')
            rows.append(dd)
        if [l for l in _ALL_LANGS if l not in used]:
            dd = W.Dropdown(options=_sec_opts(used), value=None,
                            description=f'Other dub {len(picks)+1}:', layout=W.Layout(width='320px'))
            dd.observe(_rebuild_cascade, names='value')
            rows.append(dd)
        sec_box.children = tuple(rows)
    finally:
        _rebuilding['on'] = False
lang_w.observe(_rebuild_cascade, names='value')
_rebuild_cascade()

display(lang_w, gender_w, voice_custom_w, _sec_label, sec_box, pitch_w, range_w,
        remove_voice_w, dynamic_dur_w, skip_silent_w, burn_subs_w,
        translate_title_w, hq_demucs_w, allow_no_amb_w, ambient_gain_w,
        out_mode_w, out_path_w, cache_path_w)
print()
print('Dynamic duration: voice stays natural; video stretches to match. ~3-5x slower per video.')
print('Skip silent: avoids dubbing over moments the speaker stayed quiet (Whisper hallucinations).')
print('Burn subtitles: bakes the SRT into the video (forces re-encode, slower).')
print('Cache dir: persists video/ambient per URL so other languages reuse them.')
print('Vocal removal: mdx_extra preserves workshop ambient better than the default htdemucs')
print('               but is ~3-4x slower per video. Toggle on for keepers, off for batches.')
print('Continue without ambient: by default a video is aborted if Demucs cannot')
print('                          extract the ambient stem (avoids producing dry dubs).')
print('                          Tick to override and let the video render anyway.')
print('Ambient gain: how loud the no-vocals stem sits under the TTS. 3.5 is the')
print('              default; raise for more room tone, lower if it competes with the voice.')
print()
print('Adjust the values above, then run the next cell.')
if not _drive_mounted:
    print()
    print('NOTE: Google Drive is not mounted (step 3 not run) — output defaulted to local.')
    print('      Mount Drive and re-run this cell if you want to save there instead.')

In [ ]:
# Lock in configuration
CONFIG = {
    'lang':                 lang_w.value,
    'secondary_langs':      chosen_secondaries(),
    'gender':               gender_w.value,
    'voice':                voice_custom_w.value.strip() or None,
    'pitch_st':             pitch_w.value,
    'range_expr':           range_w.value,
    'remove_voice':         remove_voice_w.value,
    'dynamic_duration':     dynamic_dur_w.value,
    'skip_silent_segments': skip_silent_w.value,
    'burn_in_subs':         burn_subs_w.value,
    'translate_titles':     translate_title_w.value,
    'demucs_model':         'mdx_extra' if hq_demucs_w.value else 'htdemucs',
    'ambient_gain':         ambient_gain_w.value,
    'allow_no_ambient':     allow_no_amb_w.value,
    'output_dir':           out_path_w.value,
    'srt_dir':              SRT_DIR,
    'cache_root':           cache_path_w.value or None,
}

url_text = (url_box.value or '').strip()
if not url_text:
    raise RuntimeError('No URLs provided. Paste in the textarea.')
URL_SOURCE = [l for l in url_text.splitlines() if l.strip()]
print(f'URL source: textarea ({len(URL_SOURCE)} lines)')

from batch_dub_edge import resolve_voice
effective_voice = resolve_voice(CONFIG['lang'], CONFIG['gender'], CONFIG['voice'])
print(f'Effective voice: {effective_voice}')
for _lg in CONFIG['secondary_langs']:
    print(f"Secondary voice [{_lg}]: {resolve_voice(_lg, CONFIG['gender'], None)} (default)")
print('\nConfiguration:')
for k, v in CONFIG.items():
    print(f'  {k:22} = {v}')

import os
if CONFIG['output_dir'].startswith('/content/drive/') and not os.path.ismount('/content/drive'):
    _fallback = f"/content/output/{CONFIG['lang']}"
    print()
    print(f'  [NOTICE] Drive is not mounted but output_dir points there.')
    print(f'           Falling back to local: {_fallback}')
    print(f'           Run step 3 before step 7 if you want Drive instead.')
    CONFIG['output_dir'] = _fallback

import batch_dub
batch_dub.AMBIENT_GAIN = CONFIG['ambient_gain']
print(f"  AMBIENT_GAIN set to {batch_dub.AMBIENT_GAIN} for this run")


## 6. Preview

In [ ]:
import pandas as pd
from batch_dub import load_urls, build_items, build_output_name

all_urls = load_urls(URL_SOURCE)
preview_urls = all_urls[:3]
preview_items = build_items(preview_urls,
                            translate_titles=CONFIG['translate_titles'],
                            target_lang=CONFIG['lang'].lower())
rows = []
for it in preview_items:
    rows.append({
        'N#': it.n,
        'Date': it.upload_date or '—',
        'Duration': f'{int(it.duration//60)}:{int(it.duration%60):02d}' if it.duration else '—',
        'Title': it.title or f'[fallback — {it.error}]',
        'Output filename': build_output_name(it, CONFIG['lang'], CONFIG['translate_titles']),
    })
pd.DataFrame(rows).style.set_properties(**{'text-align': 'left'}).hide(axis='index')

## 7. Run batch (multi-language, resilient)

Breadth-first: the **primary** language runs first over every video. If YouTube's bot wall kills downloads, it stops and dubs the **cached** videos into each **secondary** language (default voice). A real TTS failure aborts that one video rather than producing it incomplete.

In [ ]:
from batch_dub_edge import run_batch_multilang

results = run_batch_multilang(
    urls=URL_SOURCE,
    srt_dir=CONFIG['srt_dir'],
    output_dir=CONFIG['output_dir'],
    primary_lang=CONFIG['lang'],
    secondary_langs=CONFIG['secondary_langs'],
    gender=CONFIG['gender'],
    voice=CONFIG['voice'],
    pitch_st=CONFIG['pitch_st'],
    translate_titles=CONFIG['translate_titles'],
    remove_voice=CONFIG['remove_voice'],
    dynamic_duration=CONFIG['dynamic_duration'],
    skip_silent_segments=CONFIG['skip_silent_segments'],
    burn_in_subs=CONFIG['burn_in_subs'],
    cache_root=CONFIG['cache_root'],
    range_expr=CONFIG['range_expr'],
    demucs_model=CONFIG['demucs_model'],
    allow_no_ambient=CONFIG['allow_no_ambient'],
)


## 8. Download outputs (local mode only)

In [ ]:
from google.colab import files
from pathlib import Path
import os

drive_mounted = os.path.ismount('/content/drive')
n_done = n_drive = n_dl = 0
for it in results:
    if it.status != 'done' or not it.output_path:
        continue
    n_done += 1
    p = Path(it.output_path)
    if not p.exists():
        print(f'  [WARN] {p.name} marked done but file not found at {p}')
        continue
    on_drive = drive_mounted and str(p).startswith('/content/drive/')
    if on_drive:
        n_drive += 1
        print(f'  (skip) {p.name} — already in Drive')
        continue
    print(f'  Downloading {p.name}...')
    files.download(str(p))
    n_dl += 1
print(f'\n{n_done} done | {n_drive} in Drive | {n_dl} downloaded')

## Reset for next run

Wipes the cached Python modules from this session **and** the cloned repo on disk. Run this when you want the next iteration to pick up new code (after a `git pull` on the repo).

In [ ]:
# Wipe everything that gets stale between runs:
#   - cached batch_dub modules in this Python session
#   - cloned repo on disk (forces a fresh git clone next time you run cell 4)
# Run this when you want the next iteration to pick up new code.
import sys
for _m in list(sys.modules):
    if _m.startswith('batch_dub'):
        del sys.modules[_m]
print('Cleared cached batch_dub modules')

!rm -rf /content/Sta-RU
print('Removed /content/Sta-RU')

## Free disk: processing cache

Removes the cross-language cache (`/tmp/sta-ru-cache`) and per-video work dirs. Run this when you're done with a set of videos and don't plan to re-dub them in another language.

In [ ]:
import shutil, os
for p in ('/tmp/sta-ru-cache', '/tmp/sta-ru-edge', '/tmp/sta-ru-work', '/tmp/dbg', '/tmp/no_voice'):
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)
        print(f'Removed {p}')
    else:
        print(f'(skip) {p} not present')
!df -h /tmp | tail -1

## Free disk: downloaded ML models

Removes the XTTS-v2 checkpoint (~2 GB) and the Demucs htdemucs weights (~80 MB). They'll be re-downloaded the next time you run the pipeline.

In [ ]:
import shutil, os
paths = [
    '/root/.local/share/tts',                     # Coqui XTTS-v2 model
    '/root/.cache/torch/hub/checkpoints',         # Demucs + other torch hub models
    '/root/.cache/huggingface',                   # HF cache (translation, etc)
]
for p in paths:
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)
        print(f'Removed {p}')
    else:
        print(f'(skip) {p} not present')
!df -h / | tail -1